In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from adios2 import FileReader
from adios2 import Stream
import glob
import os
import re

In [ ]:
outputDir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/"

In [ ]:
def plot_neighbors(outputDir):
    
    re_runID = re.compile("run_([0-9]+).bp")
    allFiles = [int(re_runID.match(x)[1]) for x in os.listdir(outputDir) if re_runID.match(x)]
    
    min(allFiles), max(allFiles)
    
    neighbors_list = []
    temperature_list = []
    runlength = 20
    
    for index in range(min(allFiles), max(allFiles)):
        with Stream(os.path.join(outputDir,f"run_{index}.bp"), "r") as s:
            nn = []
            for step in s.steps():
                if "number of neighbors" in s.available_variables():
                    if "real temperature" not in s.available_variables():
                        raise ValueError("Require 'real temperature' where 'numer of neighbors'")
                    nn.append(s.read("number of neighbors"))
                    temperature_list.append(s.read("real temperature"))
            
            nn = np.concatenate(nn)
            nnn = nn.reshape((nn.shape[0]//runlength, runlength, nn.shape[1]))
            nn = np.mean(nnn, axis=1)
            neighbors_list.append(nn)
            
    neighbors = np.array(neighbors_list)
    temperature = np.array(temperature_list) 
                
    fig, ax1 = plt.subplots(figsize=(7, 4))
    for j in range(1, neighbors.shape[2]):
        y = neighbors[:,0,j]
        
        ax1.plot(y, label=f'shell {j}')
        ax2 = ax1.twinx()
        ax2.plot(temperature, label='temperature')
        
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Average Number of Neighbors")
    ax2.set_ylabel("Temperature")
    ax1.set_title("Neighbor Analysis")
    
    ax1.legend()
    ax2.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [ ]:
outputDir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/"
plot_neighbors(outputDir)

In [ ]:
file_path = "/home/hadis/custom_vector/buildParticleOriented/buildVS/jan28_dotTestAdios/outputs/run_01.bp"


In [ ]:
import numpy as np
from adios2 import FileReader

s = FileReader(file_path)
v = s.read("positions", step_selection=[0, 100])
s.available_variables()["positions"]

In [ ]:
v

In [ ]:
def readVariable(file_path, variable_name, start=0, end=100):
    var = FileReader(file_path).read(variable_name, step_selection=[start, end])
    return var

In [ ]:
pos = readVariable(file_path, "positions")

In [ ]:
import numpy as np
from adios2 import Stream

def read_adios_output(file_path, print_summary=True):
    
    data = {}

    with Stream(file_path, "r") as s:
        for step in s.steps():
            available_vars = s.available_variables()
            for var in available_vars:
                if var not in data:
                    data[var] = []
                data[var].append(s.read(var))

    for var in data:
        data[var] = np.array(data[var])

    if (print_summary):
        print("Data contents summary:\n----------------------")
        for idx, (var, values) in enumerate(data.items(), start=0):
            if values is not None:
                shape = values.shape
                if len(shape) == 1:
                    description = shape[0]
                else:
                    description = f"{shape[0]}*{list(shape[1:])}"
                print(f"{idx}. {var:<40} {description}")

    return list(data.values())


In [ ]:
simulation_data = read_adios_output(file_path)


In [ ]:
import matplotlib.pyplot as plt

def plot_snapshot_configuration(step, variable):
    all_steps = variable.shape[0]
    fig, ax = plt.subplots()

    ax.scatter(variable[step, :, 0], variable[step, :, 1], label="initial configuration")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 20)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"configuration (step {step} of {all_steps})")
    plt.show()
    
def plot_energies(kinetic_energy, potential_energy):
    fig, ax = plt.subplots()

    ax.plot(kinetic_energy[:, 0], label="kinetic energy")
    ax.plot(potential_energy[:]/100, label="potential energy")
    ax.plot(kinetic_energy[:, 0, 0] + kinetic_energy[:, 0, 1] + potential_energy[:]/100, label="total energy")
    ax.plot(kinetic_energy[:, 0, 0] + kinetic_energy[:, 0, 1], label="linear kinetic energy")
    ax.set_xlabel("step")
    ax.set_ylabel("energy")
    ax.set_title("Energies")
    ax.legend()
    plt.show()


In [ ]:
plot_snapshot_configuration(-1, simulation_data[3])

In [ ]:
plot_snapshot_configuration(0, simulation_data[3])

In [ ]:
plot_snapshot_configuration(0, simulation_data[0])

In [ ]:
plot_energies(simulation_data[7], simulation_data[9])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def plot_neighbors(neighbor_list, smoothing=True, window_length=7, polyorder=2):
    
    fig, ax = plt.subplots(figsize=(7, 4))
    for j in range(1, neighbor_list.shape[2]):
        y = neighbor_list[:,0,j]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        ax.plot(y, label=f'shell {j}')
        
    ax.set_xlabel("Step")
    ax.set_ylabel("Average Number of Neighbors")
    ax.set_title("Neighbor Analysis")
    
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

def plot_temperature (temperature, raw_temperature):
    fig, ax = plt.subplots()

    ax.plot(temperature, label="temperature")
    ax.plot(raw_temperature, label="raw temperature")
    ax.set_xlabel("step")
    ax.set_ylabel("temperature")
    ax.set_title("Temperature")
    ax.legend()
    plt.show()

In [ ]:
plot_neighbors(simulation_data[6])

In [ ]:
def plot_com_velocity(com_linear_velocity):
    fig, ax = plt.subplots()

    ax.plot(com_linear_velocity[:, 0, 0], label="Vx")
    ax.plot(com_linear_velocity[:, 0, 1], label="Vy")
        
    ax.set_xlabel("step")
    ax.set_ylabel("velocity")
    ax.set_title("Center of Mass velocity")
    ax.legend()
    plt.show()
    
def plot_com_angular_velocity(com_angular_velocity):
    fig, ax = plt.subplots()

    ax.plot(com_angular_velocity)
    
    ax.set_xlabel("step")
    ax.set_ylabel("angular velocity")
    ax.set_title("Center of Mass angular velocity")
    plt.show()
    
def plot_average_velocities(velocities):
    fig, ax = plt.subplots()

    ax.plot(np.average(velocities[:, :, 0], axis=1), label="Vx")
    ax.plot(np.average(velocities[:, :, 1], axis=1), label="Vy")
        
    ax.set_xlabel("step")
    ax.set_ylabel("velocity")
    ax.set_title("Center of Mass velocity")
    ax.legend()
    plt.show()   

In [ ]:
plot_average_velocities(simulation_data[2])

In [ ]:
plot_com_velocity(simulation_data[4])
plot_com_angular_velocity(simulation_data[3])